# Marketplace Listing Integrity Agent

A hybrid pipeline that decides whether a marketplace listing's image genuinely matches its text description — grounded by a **RAG** knowledge base of category-specific verification policy, with an **LLM** reserved for the small number of steps that are genuine judgment calls, not every step.

### Why this isn't a full ReAct loop end to end
The first version of this agent put every step behind an LLM decision — detect category, detect product, decide to retrieve a policy, decide to check a feature, decide when to stop. Testing surfaced two problems: it took ~1-2 minutes per listing locally (8-12 sequential LLM turns, each re-processing a growing context), and most of those turns weren't actually judgment calls — "call detect_category, then detect_product" is the same fixed sequence every time. Worse, making OCR/VQA fallback an LLM *choice* meant the model sometimes skipped it entirely, "inferring" an unconfirmed brand from context instead of checking — exactly the failure mode this project exists to catch.

This version keeps the LLM only where testing showed it's genuinely needed. Everything else is deterministic Python, the same way the sibling ridesharing project's own Stage 1 handled its rule-based logic.

### Pipeline at a glance

| Stage | What happens | Needs an LLM? |
|---|---|---|
| 1. Detect category, then product | two short constrained VQA questions | No — fixed order, no judgment |
| 2. Self-consistency check + retry | does the product belong to the category? | No — a membership check + a fixed one-time retry rule |
| 3. Hard-stop vs. description | does the image-grounded product agree with the claim? | No — normalized string comparison |
| 4. Retrieve category policy | RAG: exact match, semantic search as fallback | No — embedding model only |
| 5. Build checklist | policy's critical features + any other concrete claim | No — set logic |
| 6. Verify each feature | OCR first, then a fixed-template VQA fallback question | No — deterministic matching |
| 7. Resolve anything still ambiguous | **only if step 6 left something unresolved** | **Yes — the only LLM involvement** |
| 8. Final verdict | hard veto on any critical mismatch | No — computed, not generated |

A listing where every feature resolves cleanly in step 6 never invokes the LLM at all.

> **Requires:** a local [Ollama](https://ollama.com) server running `qwen2.5:14b` (only used if escalation happens) and `nomic-embed-text` (embeddings). See the project README for setup.

## Setup

In [ ]:
import sys
import os
import time
sys.path.append(os.path.abspath(".."))

from src.config import OLLAMA_MODEL, EMBED_MODEL
from src.policy_corpus import CATEGORY_POLICIES
from src.retrieval import build_policy_corpus, retrieve_category_policy, get_checklist_features
from src.tools import ALL_TOOLS
from src.vision_tools import detect_category_and_product
from src.agent import build_graph, verify_listing, _build_checklist, _deterministic_verify
from src.listings import LISTINGS

print(f"Reasoning model : {OLLAMA_MODEL}  (only loaded if a listing needs escalation)")
print(f"Embedding model : {EMBED_MODEL}")
print(f"LLM-facing tools: {[t.name for t in ALL_TOOLS]}  (down from 5 in the full-ReAct version)")

---
## Step 1: Building the Category Policy Knowledge Base (RAG)

Unchanged from the full-ReAct version — each category still gets a policy document with critical/cosmetic features, embedded into Chroma. What's different: `retrieve_category_policy` is now a plain function the pipeline always calls once category is known, not a tool an LLM decides to call.

In [ ]:
vectorstore = build_policy_corpus()
print(f"Collection size: {len(vectorstore.get()['ids'])} documents")

for category in ["grocery", "a pair of wireless bluetooth headphones", "a cotton crew-neck t-shirt"]:
    text = retrieve_category_policy(category)
    resolved = text.split("Category:")[1].split("(")[0].strip().rstrip(".")
    print(f"'{category}' -> {resolved}")

---
## Step 2: Deterministic Category & Product Detection

`detect_category_and_product` runs both VQA questions, checks whether the answers are self-consistent, and retries once if not — all in plain code. No LLM reasoning turn happens here; there's nothing to reason about, just two tool calls and a membership check.

In [ ]:
listing = LISTINGS[0]
print(f"Listing: {listing['name']}\n")

t0 = time.time()
category, product, product_confirmed = detect_category_and_product(listing["image_url"])
print(f"category           : {category}")
print(f"product            : {product}")
print(f"product_confirmed  : {product_confirmed}")
print(f"(took {time.time() - t0:.1f}s, zero LLM reasoning calls)")

---
## Step 3: Deterministic Feature Verification

For each checklist feature: try OCR first (`check_against_ocr`), fall back to a fixed-template VQA question if OCR doesn't find it (`check_against_vqa_answer`). Both comparisons are normalized string matching — same mechanism that fixed the plural/singular and "guessing at unstructured OCR text" bugs found during testing, now applied deterministically instead of hoping a prompt instruction gets followed.

In [ ]:
critical_features, cosmetic_features = get_checklist_features(category)
checklist, checklist_critical = _build_checklist(critical_features, listing["description_features"])
print(f"Checklist: {checklist}\n")

t0 = time.time()
results, ambiguous = _deterministic_verify(listing["image_url"], checklist)
for feature, r in results.items():
    status = "match" if r["match"] is True else "mismatch" if r["match"] is False else "unconfirmed"
    print(f"  {feature:<10} [{r['source']:<3}] {status:<12} (claimed: '{r['value']}')")
print(f"\nAmbiguous (needs LLM escalation): {ambiguous or 'none'}")
print(f"(took {time.time() - t0:.1f}s, zero LLM reasoning calls so far)")

---
## Step 4: The Narrow Escalation Sub-Graph

Standard LangGraph ReAct shape (`agent` <-> `tools` via `tools_condition`) — but bound to only `read_label_text` and `ask_vision_question`, and only ever invoked when Step 3 leaves something ambiguous. This is the only part of the pipeline that's still genuinely agentic, and it's reached far less often than every step was in the full-ReAct version.

In [ ]:
graph = build_graph()

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("Diagram rendering unavailable, printing graph structure instead:")
    print(graph.get_graph().draw_mermaid())

---
## Step 5: Running the Full Pipeline

Now the whole thing end to end via `verify_listing`, across all 5 listings — a correct one, three with injected mismatches (wrong brand, wrong size, wrong product entirely), and one more correct listing as a control. Watch the `escalated` field and the timing: listings that resolve cleanly should be dramatically faster than the full-ReAct version's ~1-2 min/listing, since most of them now make zero LLM calls at all.

In [ ]:
for listing in LISTINGS:
    print(f"Listing: {listing['name']}")
    t0 = time.time()
    result = verify_listing(graph, listing["image_url"], listing["description_features"])
    elapsed = time.time() - t0

    print(f"  category    : {result['category']}")
    print(f"  hard_stop   : {result['hard_stop']}")
    print(f"  escalated   : {result['escalated']}  (ambiguous features: {result['ambiguous_features'] or 'none'})")
    print(f"  took        : {elapsed:.1f}s")
    print()
    print(result["answer"])
    print("\n" + "=" * 70 + "\n")

---
## Summary

### What changed from the full-ReAct version
The pipeline used to put every decision behind an LLM call — ~8-12 sequential reasoning turns per listing, most of them not actually judgment calls. This version:

- **Runs category/product detection, the self-consistency check, the hard-stop comparison, policy retrieval, and checklist building entirely as plain code** — no LLM involved, because none of these steps were ever genuinely ambiguous
- **Verifies each feature deterministically** — OCR first, a fixed-template VQA question as fallback, normalized string matching for both — the exact mechanism that fixed the "8g cup" and plural/singular bugs found during testing, now structurally guaranteed rather than prompt-requested
- **Escalates to a narrow LLM ReAct loop only for features that stay genuinely ambiguous** after the deterministic pass — most listings never reach this step at all
- **Computes the verdict and writes the explanation from code**, not generation — the hard-veto logic and the "unconfirmed ≠ match" rule are now enforced by construction, not by hoping the model remembers a prompt instruction

### Key findings
- The specific bug that let a wrong-brand listing pass as "Likely match" — the LLM skipping `ask_vision_question` and inferring an answer instead — is now structurally impossible: OCR-then-VQA-fallback always runs in code, there's no LLM choice to skip it
- The plural/singular false-mismatch bug ("chicken breast" vs. "chicken breasts") is fixed by normalization applied uniformly, not by a prompt rule a smaller model could ignore
- Escalation frequency itself is a useful signal — a category/listing type that escalates often is telling you its `FEATURE_QUESTION_TEMPLATES` or `common_products` list needs work, which is a much more actionable signal than "the agent got this one wrong"

### Limitations
- **The hard-stop and hierarchy checks are deterministic with no escalation path** — a genuinely ambiguous product/category disagreement is resolved by a fixed rule (trust category over an unconfirmed product), not investigated further, unlike feature-level ambiguity
- **`FEATURE_QUESTION_TEMPLATES` and `common_products` are hand-curated and category-agnostic where they probably shouldn't be** — a real system would likely need per-category question templates, not one generic template per feature name reused everywhere
- **Live testing is still grocery-only** — same gap as before, unrelated to this redesign
- **No human-in-the-loop yet** — every verdict is fully automatic, same open item as before

### Potential future work
- Measure escalation rate across a larger, more varied test set — that rate is the real scalability metric for this design, more useful than per-listing latency alone
- Per-category VQA question templates instead of one generic template per feature name
- A human-in-the-loop gate before a "Likely mismatch" verdict executes, for high-risk categories or sellers
- Real verified images for electronics and apparel to close the live-testing gap